# 🕋 Multi-Scholar Fatwa Candidate Generation (6,000 Candidates)
### 3,000 Official Main Candidates + 3,000 Backup Negative Pool

This notebook runs on **Kaggle GPU (T4 x2 or P100)** to retrieve and rerank candidate fatwas across **7 Sunni / Salafi authorities** for 250 curated jurisprudential questions.

---
### ⚙️ Kaggle Instructions (Before Running):
1. **Enable Accelerator**: In the right sidebar, set **Accelerator** to **GPU T4 x2** or **GPU P100**.
2. **Enable Internet**: In the right sidebar, verify **Internet ON** is enabled (needed to download Hugging Face models & embeddings).
3. Click **Run All** (Total expected runtime: ~3 to 5 minutes).

---
### 📦 Output Deliverables in `/kaggle/working/`:
- `candidates_official_3k.csv` & `.json`: Exactly 3,000 top candidate fatwas (the official pool to label locally).
- `candidates_backup_3k.csv` & `.json`: Exactly 3,000 lower-ranked candidate fatwas (the negative distractor reserve).
- `candidates_all_6k.csv` & `.json`: Combined dataset of all 6,000 candidates with `dataset_split` and `rank_within_scholar` tags.
- `candidates_6k_bundle.zip`: One-click downloadable archive containing all CSV and JSON files.


In [ ]:
# 1. Install & Verify Dependencies
# Note: Kaggle already has torch, transformers, scipy, pandas, and numpy pre-installed.
# We only install sentence-transformers and python-docx without -U to protect Kaggle's pre-compiled C-extensions.
!pip install -q sentence-transformers python-docx

import os
import sys
import time
import json
import zipfile
from pathlib import Path

# Self-healing check: if numpy was already broken by a previous pip upgrade in this session:
try:
    import numpy as np
    from scipy.sparse import coo_matrix
except ImportError:
    print("\n! Detected NumPy / Scipy C-extension mismatch from an earlier pip install.")
    print("! Please click in Kaggle: [Session] -> [Restart Session] (or the ↺ button in the top bar) and re-run.\n")

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print("=" * 60)
print("HARDWARE ACCELERATION STATUS")
print("=" * 60)
cuda_ok = torch.cuda.is_available()
print(f"CUDA Available: {cuda_ok}")
if cuda_ok:
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:     {vram_gb:.2f} GB")
else:
    print("WARNING: CUDA not detected! Please turn on GPU in Kaggle settings.")
print("=" * 60)


In [ ]:
# 2. Download Multi-Scholar Corpus & Precomputed Embeddings from Hugging Face
from huggingface_hub import hf_hub_download

DATA_REPO_ID = "ZenoAbderrahim/binbaz_fatwas_rag"
DATA_DIR = Path("/kaggle/working/data") if Path("/kaggle/working").exists() else Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES_TO_DOWNLOAD = [
    "multi_scholar_clean.csv",
    "qwen3_06b_multi_scholar_embeddings.pt",
    "multi_scholar_metadata.json"
]

print(f"Syncing corpus files from Hugging Face dataset: '{DATA_REPO_ID}'...")
for fname in FILES_TO_DOWNLOAD:
    dest = DATA_DIR / fname
    if not dest.exists():
        print(f"  • Downloading {fname} ...")
        hf_hub_download(repo_id=DATA_REPO_ID, repo_type="dataset", filename=fname, local_dir=DATA_DIR)
    else:
        print(f"  ✓ {fname} already present.")

print("\nLoading cleaned fatwa corpus...")
corpus_df = pd.read_csv(DATA_DIR / "multi_scholar_clean.csv")
print(f"✓ Total corpus size: {len(corpus_df):,} fatwas across {corpus_df['source'].nunique()} scholars:")
print(corpus_df['source'].value_counts().to_string())

print("\nLoading precomputed Qwen 0.6B corpus embeddings...")
corpus_embeddings = torch.load(DATA_DIR / "qwen3_06b_multi_scholar_embeddings.pt", map_location="cpu")
print(f"✓ Corpus embeddings shape: {corpus_embeddings.shape}")


In [ ]:
# 3. Load the 250 Jurisprudential Questions
# Priority: 1. Kaggle Input CSV/DOCX if attached | 2. Pre-embedded verified Python dataset

def find_input_file(filename: str) -> Path | None:
    for p in Path("/kaggle/input").rglob(filename) if Path("/kaggle/input").exists() else []:
        return p
    for p in Path(".").rglob(filename):
        return p
    return None

questions_data = None

# Check for CSV input
csv_path = find_input_file("250_fatwa_relevance_questions.csv")
if csv_path and csv_path.exists():
    print(f"✓ Loading questions from attached input CSV: {csv_path}")
    questions_df = pd.read_csv(csv_path)
    questions_data = questions_df.to_dict(orient="records")
else:
    print("Loading verified 250 questions from pre-extracted dataset...")
    # Pre-extracted from 250_fatwa_relevance_questions.docx via code
    raw_questions = [
  {
    "question_id": 1,
    "category": "Aqidah and Tawhid",
    "question": "ما معنى توحيد الربوبية؟"
  },
  {
    "question_id": 2,
    "category": "Aqidah and Tawhid",
    "question": "ما الفرق بين توحيد الربوبية وتوحيد الألوهية؟"
  },
  {
    "question_id": 3,
    "category": "Aqidah and Tawhid",
    "question": "ما معنى توحيد الأسماء والصفات؟"
  },
  {
    "question_id": 4,
    "category": "Aqidah and Tawhid",
    "question": "ما شروط لا إله إلا الله؟"
  },
  {
    "question_id": 5,
    "category": "Aqidah and Tawhid",
    "question": "ما معنى شهادة أن محمداً رسول الله؟"
  },
  {
    "question_id": 6,
    "category": "Aqidah and Tawhid",
    "question": "ما الفرق بين الشرك الأكبر والشرك الأصغر؟"
  },
  {
    "question_id": 7,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم دعاء الأموات وطلب المدد منهم؟"
  },
  {
    "question_id": 8,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم الذبح لغير الله؟"
  },
  {
    "question_id": 9,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم الحلف بغير الله؟"
  },
  {
    "question_id": 10,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم تعليق التمائم لدفع العين؟"
  },
  {
    "question_id": 11,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم الرقية التي تحتوي على ألفاظ مجهولة؟"
  },
  {
    "question_id": 12,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم الاستعانة بالجن في العلاج؟"
  },
  {
    "question_id": 13,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم قول: ما شاء الله وشئت؟"
  },
  {
    "question_id": 14,
    "category": "Aqidah and Tawhid",
    "question": "هل يجوز طلب الدعاء من شخص ميت؟"
  },
  {
    "question_id": 15,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم زيارة القبور؟"
  },
  {
    "question_id": 16,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم شد الرحال إلى القبور للتبرك؟"
  },
  {
    "question_id": 17,
    "category": "Aqidah and Tawhid",
    "question": "ما الفرق بين التوسل المشروع والتوسل الممنوع؟"
  },
  {
    "question_id": 18,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم التوسل بجاه النبي صلى الله عليه وسلم؟"
  },
  {
    "question_id": 19,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم الاستغاثة بالنبي صلى الله عليه وسلم بعد وفاته؟"
  },
  {
    "question_id": 20,
    "category": "Aqidah and Tawhid",
    "question": "كيف يجمع المسلم بين إثبات الصفات وتنزيه الله عن مشابهة المخلوقين؟"
  },
  {
    "question_id": 21,
    "category": "Aqidah and Tawhid",
    "question": "ما معنى الإيمان بأسماء الله وصفاته دون تحريف ولا تعطيل؟"
  },
  {
    "question_id": 22,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم إنكار صفة ثابتة في القرآن أو السنة؟"
  },
  {
    "question_id": 23,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم سب الدين أو الاستهزاء بشيء من شعائر الإسلام؟"
  },
  {
    "question_id": 24,
    "category": "Aqidah and Tawhid",
    "question": "ما ضابط تكفير المسلم المعين؟"
  },
  {
    "question_id": 25,
    "category": "Aqidah and Tawhid",
    "question": "ما الفرق بين الكفر الأكبر والكفر الأصغر؟"
  },
  {
    "question_id": 26,
    "category": "Aqidah and Tawhid",
    "question": "هل كل من وقع في الشرك الأكبر يحكم عليه مباشرة بأنه مشرك بعينه؟"
  },
  {
    "question_id": 27,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم تعليق آيات القرآن في السيارة للحفظ؟"
  },
  {
    "question_id": 28,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم التبرك بالأشجار والأحجار؟"
  },
  {
    "question_id": 29,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم الذهاب إلى الكهان والعرافين؟"
  },
  {
    "question_id": 30,
    "category": "Aqidah and Tawhid",
    "question": "ما حكم تصديق الأبراج والتوقعات الفلكية؟"
  },
  {
    "question_id": 31,
    "category": "Prayer and purification",
    "question": "ما شروط صحة الصلاة؟"
  },
  {
    "question_id": 32,
    "category": "Prayer and purification",
    "question": "ما أركان الصلاة التي لا تصح بدونها؟"
  },
  {
    "question_id": 33,
    "category": "Prayer and purification",
    "question": "ما حكم من ترك الصلاة تكاسلاً؟"
  },
  {
    "question_id": 34,
    "category": "Prayer and purification",
    "question": "ما حكم تأخير الصلاة عن وقتها بسبب العمل؟"
  },
  {
    "question_id": 35,
    "category": "Prayer and purification",
    "question": "ما حكم الصلاة مع وجود نجاسة على الثوب؟"
  },
  {
    "question_id": 36,
    "category": "Prayer and purification",
    "question": "ماذا يفعل من شك في عدد الركعات؟"
  },
  {
    "question_id": 37,
    "category": "Prayer and purification",
    "question": "ما حكم الوسوسة في الوضوء؟"
  },
  {
    "question_id": 38,
    "category": "Prayer and purification",
    "question": "هل ينتقض الوضوء بلمس المرأة؟"
  },
  {
    "question_id": 39,
    "category": "Prayer and purification",
    "question": "هل ينتقض الوضوء بخروج الدم؟"
  },
  {
    "question_id": 40,
    "category": "Prayer and purification",
    "question": "ما حكم المسح على الجوربين؟"
  },
  {
    "question_id": 41,
    "category": "Prayer and purification",
    "question": "ما شروط المسح على الخفين؟"
  },
  {
    "question_id": 42,
    "category": "Prayer and purification",
    "question": "ما حكم من نام جالساً ثم صلى دون وضوء؟"
  },
  {
    "question_id": 43,
    "category": "Prayer and purification",
    "question": "هل يجب الترتيب في الوضوء؟"
  },
  {
    "question_id": 44,
    "category": "Prayer and purification",
    "question": "ما حكم ترك التسمية قبل الوضوء نسياناً؟"
  },
  {
    "question_id": 45,
    "category": "Prayer and purification",
    "question": "كيف يتطهر من عجز عن استعمال الماء؟"
  },
  {
    "question_id": 46,
    "category": "Prayer and purification",
    "question": "متى يجوز التيمم مع وجود الماء؟"
  },
  {
    "question_id": 47,
    "category": "Prayer and purification",
    "question": "ما حكم الجمع بين الصلاتين بسبب المطر؟"
  },
  {
    "question_id": 48,
    "category": "Prayer and purification",
    "question": "ما حكم قصر الصلاة للمسافر؟"
  },
  {
    "question_id": 49,
    "category": "Prayer and purification",
    "question": "ما المسافة التي يجوز فيها قصر الصلاة؟"
  },
  {
    "question_id": 50,
    "category": "Prayer and purification",
    "question": "متى يبدأ المسافر بالقصر ومتى ينتهي؟"
  },
  {
    "question_id": 51,
    "category": "Prayer and purification",
    "question": "هل يجوز للمسافر أن يتم الصلاة خلف إمام مقيم؟"
  },
  {
    "question_id": 52,
    "category": "Prayer and purification",
    "question": "ما حكم صلاة الجماعة في المسجد للرجال؟"
  },
  {
    "question_id": 53,
    "category": "Prayer and purification",
    "question": "ما حكم من أدرك الإمام في الركوع؟"
  },
  {
    "question_id": 54,
    "category": "Prayer and purification",
    "question": "كيف يقضي المسبوق ما فاته من الصلاة؟"
  },
  {
    "question_id": 55,
    "category": "Prayer and purification",
    "question": "ما حكم رفع اليدين عند الركوع والرفع منه؟"
  },
  {
    "question_id": 56,
    "category": "Prayer and purification",
    "question": "ما حكم قراءة الفاتحة للمأموم خلف الإمام؟"
  },
  {
    "question_id": 57,
    "category": "Prayer and purification",
    "question": "أين يضع المصلي يديه في القيام؟"
  },
  {
    "question_id": 58,
    "category": "Prayer and purification",
    "question": "ما حكم تحريك السبابة في التشهد؟"
  },
  {
    "question_id": 59,
    "category": "Prayer and purification",
    "question": "ما حكم الصلاة على النبي صلى الله عليه وسلم في التشهد الأخير؟"
  },
  {
    "question_id": 60,
    "category": "Prayer and purification",
    "question": "ما حكم سجود السهو إذا نسي المصلي واجباً؟"
  },
  {
    "question_id": 61,
    "category": "Prayer and purification",
    "question": "ماذا يفعل من شك بعد الصلاة في أنه ترك ركناً؟"
  },
  {
    "question_id": 62,
    "category": "Prayer and purification",
    "question": "ما حكم الصلاة في مكان فيه صور؟"
  },
  {
    "question_id": 63,
    "category": "Prayer and purification",
    "question": "هل تصح الصلاة خلف إمام يقع في بدعة؟"
  },
  {
    "question_id": 64,
    "category": "Prayer and purification",
    "question": "ما حكم صلاة الوتر وكم أقل عدد ركعاتها؟"
  },
  {
    "question_id": 65,
    "category": "Prayer and purification",
    "question": "ما أفضل وقت لصلاة الوتر؟"
  },
  {
    "question_id": 66,
    "category": "Prayer and purification",
    "question": "ما حكم صلاة الضحى؟"
  },
  {
    "question_id": 67,
    "category": "Prayer and purification",
    "question": "هل تقضي المرأة الصلاة التي تركتها أثناء الحيض؟"
  },
  {
    "question_id": 68,
    "category": "Prayer and purification",
    "question": "ما حكم الصلاة مع مدافعة البول أو الغائط؟"
  },
  {
    "question_id": 69,
    "category": "Prayer and purification",
    "question": "ما حكم الصلاة في الثوب الضيق؟"
  },
  {
    "question_id": 70,
    "category": "Prayer and purification",
    "question": "هل يجوز المرور أمام المصلي؟"
  },
  {
    "question_id": 71,
    "category": "Fasting, zakat, and hajj",
    "question": "ما مبطلات الصيام؟"
  },
  {
    "question_id": 72,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يفطر الصائم إذا أكل ناسياً؟"
  },
  {
    "question_id": 73,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم استعمال بخاخ الربو في نهار رمضان؟"
  },
  {
    "question_id": 74,
    "category": "Fasting, zakat, and hajj",
    "question": "هل قطرة العين تفطر الصائم؟"
  },
  {
    "question_id": 75,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم استعمال معجون الأسنان أثناء الصيام؟"
  },
  {
    "question_id": 76,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم من جامع زوجته في نهار رمضان؟"
  },
  {
    "question_id": 77,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم من أفطر في رمضان بسبب المرض؟"
  },
  {
    "question_id": 78,
    "category": "Fasting, zakat, and hajj",
    "question": "متى يجب على المريض القضاء ومتى تكون عليه الفدية؟"
  },
  {
    "question_id": 79,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم صيام من نام معظم النهار؟"
  },
  {
    "question_id": 80,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز صيام يوم الجمعة منفرداً؟"
  },
  {
    "question_id": 81,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم صيام يوم عرفة لغير الحاج؟"
  },
  {
    "question_id": 82,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم صيام عاشوراء دون صيام يوم قبله أو بعده؟"
  },
  {
    "question_id": 83,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم صيام الست من شوال قبل قضاء رمضان؟"
  },
  {
    "question_id": 84,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم من أخر قضاء رمضان حتى دخل رمضان التالي؟"
  },
  {
    "question_id": 85,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز للحامل والمرضع الإفطار في رمضان؟"
  },
  {
    "question_id": 86,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم السحور بعد أذان الفجر ظناً أن الوقت لم يدخل؟"
  },
  {
    "question_id": 87,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم الإفطار اعتماداً على أذان المغرب في بلد آخر؟"
  },
  {
    "question_id": 88,
    "category": "Fasting, zakat, and hajj",
    "question": "على من تجب زكاة المال؟"
  },
  {
    "question_id": 89,
    "category": "Fasting, zakat, and hajj",
    "question": "ما النصاب الواجب في زكاة النقود؟"
  },
  {
    "question_id": 90,
    "category": "Fasting, zakat, and hajj",
    "question": "كيف تحسب زكاة الذهب المدخر للزينة؟"
  },
  {
    "question_id": 91,
    "category": "Fasting, zakat, and hajj",
    "question": "هل تجب الزكاة في مال الدين؟"
  },
  {
    "question_id": 92,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم إخراج الزكاة للأخ الفقير؟"
  },
  {
    "question_id": 93,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز دفع الزكاة لبناء مسجد؟"
  },
  {
    "question_id": 94,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز إخراج زكاة الفطر نقداً؟"
  },
  {
    "question_id": 95,
    "category": "Fasting, zakat, and hajj",
    "question": "متى يخرج المسلم زكاة الفطر؟"
  },
  {
    "question_id": 96,
    "category": "Fasting, zakat, and hajj",
    "question": "هل تجب زكاة الفطر على الجنين؟"
  },
  {
    "question_id": 97,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم تأخير زكاة المال بعد وجوبها؟"
  },
  {
    "question_id": 98,
    "category": "Fasting, zakat, and hajj",
    "question": "ما شروط وجوب الحج؟"
  },
  {
    "question_id": 99,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز الحج بمال اقترضه المسلم؟"
  },
  {
    "question_id": 100,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم حج المرأة دون محرم؟"
  },
  {
    "question_id": 101,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم من ترك المبيت بمزدلفة؟"
  },
  {
    "question_id": 102,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم من قدم السعي على الطواف في الحج؟"
  },
  {
    "question_id": 103,
    "category": "Fasting, zakat, and hajj",
    "question": "ماذا يلزم من ترك واجباً من واجبات الحج؟"
  },
  {
    "question_id": 104,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم من تجاوز الميقات دون إحرام؟"
  },
  {
    "question_id": 105,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز للمحرم استعمال الصابون المعطر؟"
  },
  {
    "question_id": 106,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم قص الشعر أو الأظافر للمحرم؟"
  },
  {
    "question_id": 107,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز للمرأة أن تستعمل النقاب والقفازين أثناء الإحرام؟"
  },
  {
    "question_id": 108,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم النيابة في الحج عن الميت؟"
  },
  {
    "question_id": 109,
    "category": "Fasting, zakat, and hajj",
    "question": "ما حكم تكرار العمرة في السفرة الواحدة؟"
  },
  {
    "question_id": 110,
    "category": "Fasting, zakat, and hajj",
    "question": "هل يجوز تأخير طواف الإفاضة إلى وقت السفر؟"
  },
  {
    "question_id": 111,
    "category": "Marriage and family",
    "question": "ما شروط صحة عقد الزواج؟"
  },
  {
    "question_id": 112,
    "category": "Marriage and family",
    "question": "هل يشترط رضا المرأة لصحة الزواج؟"
  },
  {
    "question_id": 113,
    "category": "Marriage and family",
    "question": "ما حكم الزواج دون ولي؟"
  },
  {
    "question_id": 114,
    "category": "Marriage and family",
    "question": "ما حكم إجبار الفتاة على الزواج؟"
  },
  {
    "question_id": 115,
    "category": "Marriage and family",
    "question": "ما حكم تأخير الزواج بسبب الدراسة؟"
  },
  {
    "question_id": 116,
    "category": "Marriage and family",
    "question": "ما حكم النظر إلى المخطوبة قبل العقد؟"
  },
  {
    "question_id": 117,
    "category": "Marriage and family",
    "question": "ما حدود الكلام بين الخاطب والمخطوبة؟"
  },
  {
    "question_id": 118,
    "category": "Marriage and family",
    "question": "هل يجوز للخاطب الخلوة بمخطوبته؟"
  },
  {
    "question_id": 119,
    "category": "Marriage and family",
    "question": "ما حكم اشتراط الزوجة إكمال دراستها في عقد الزواج؟"
  },
  {
    "question_id": 120,
    "category": "Marriage and family",
    "question": "ما حكم منع الزوج زوجته من زيارة أهلها؟"
  },
  {
    "question_id": 121,
    "category": "Marriage and family",
    "question": "ما حقوق الزوجة على زوجها؟"
  },
  {
    "question_id": 122,
    "category": "Marriage and family",
    "question": "ما حقوق الزوج على زوجته؟"
  },
  {
    "question_id": 123,
    "category": "Marriage and family",
    "question": "ما حكم امتناع الزوج عن النفقة مع قدرته؟"
  },
  {
    "question_id": 124,
    "category": "Marriage and family",
    "question": "هل يجب على الزوج علاج زوجته ودفع تكاليف علاجها؟"
  },
  {
    "question_id": 125,
    "category": "Marriage and family",
    "question": "ما حكم أخذ الزوج من مال زوجته دون رضاها؟"
  },
  {
    "question_id": 126,
    "category": "Marriage and family",
    "question": "هل يجوز للزوجة العمل دون إذن زوجها؟"
  },
  {
    "question_id": 127,
    "category": "Marriage and family",
    "question": "ما حكم سفر المرأة وحدها للدراسة أو العمل؟"
  },
  {
    "question_id": 128,
    "category": "Marriage and family",
    "question": "ما حكم تنظيم النسل باستعمال موانع الحمل؟"
  },
  {
    "question_id": 129,
    "category": "Marriage and family",
    "question": "ما حكم تحديد النسل نهائياً دون ضرورة؟"
  },
  {
    "question_id": 130,
    "category": "Marriage and family",
    "question": "ما حكم الطلاق في حال الغضب الشديد؟"
  },
  {
    "question_id": 131,
    "category": "Marriage and family",
    "question": "ما حكم الطلاق المعلق على شرط؟"
  },
  {
    "question_id": 132,
    "category": "Marriage and family",
    "question": "هل يقع الطلاق بالكتابة أو الرسائل؟"
  },
  {
    "question_id": 133,
    "category": "Marriage and family",
    "question": "ما حكم الطلاق الثلاث بلفظ واحد؟"
  },
  {
    "question_id": 134,
    "category": "Marriage and family",
    "question": "ما عدة المرأة المطلقة قبل الدخول؟"
  },
  {
    "question_id": 135,
    "category": "Marriage and family",
    "question": "ما عدة المرأة التي تحيض بعد الطلاق؟"
  },
  {
    "question_id": 136,
    "category": "Marriage and family",
    "question": "ما عدة المرأة التي لا تحيض؟"
  },
  {
    "question_id": 137,
    "category": "Marriage and family",
    "question": "ما عدة المتوفى عنها زوجها؟"
  },
  {
    "question_id": 138,
    "category": "Marriage and family",
    "question": "ما حكم رجعة الزوج لزوجته أثناء العدة؟"
  },
  {
    "question_id": 139,
    "category": "Marriage and family",
    "question": "ما حكم الخلع إذا كرهت المرأة الحياة مع زوجها؟"
  },
  {
    "question_id": 140,
    "category": "Marriage and family",
    "question": "هل يجوز للزوجة طلب الطلاق بسبب الضرر؟"
  },
  {
    "question_id": 141,
    "category": "Marriage and family",
    "question": "ما حكم حضانة الأطفال بعد الطلاق؟"
  },
  {
    "question_id": 142,
    "category": "Marriage and family",
    "question": "من أحق بحضانة الطفل إذا تزوجت أمه؟"
  },
  {
    "question_id": 143,
    "category": "Marriage and family",
    "question": "ما حكم منع الأب من رؤية أولاده بعد الطلاق؟"
  },
  {
    "question_id": 144,
    "category": "Marriage and family",
    "question": "ما حكم تسمية المولود باسم محرم المعنى؟"
  },
  {
    "question_id": 145,
    "category": "Marriage and family",
    "question": "ما حكم العقيقة عن المولود؟"
  },
  {
    "question_id": 146,
    "category": "Marriage and family",
    "question": "هل يجوز حلق شعر المولود والتصدق بوزنه فضة؟"
  },
  {
    "question_id": 147,
    "category": "Marriage and family",
    "question": "ما حكم عقوق الوالدين إذا أمرا بمعصية؟"
  },
  {
    "question_id": 148,
    "category": "Marriage and family",
    "question": "هل يجب طاعة الوالدين في اختيار التخصص أو الوظيفة؟"
  },
  {
    "question_id": 149,
    "category": "Marriage and family",
    "question": "ما حكم قطع الرحم بسبب أذى الأقارب؟"
  },
  {
    "question_id": 150,
    "category": "Marriage and family",
    "question": "ما حكم هجر أحد الوالدين بسبب خلاف عائلي؟"
  },
  {
    "question_id": 151,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل في بنك ربوي؟"
  },
  {
    "question_id": 152,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل في شركة تتعامل بالربا مع كون عملي مباحاً؟"
  },
  {
    "question_id": 153,
    "category": "Business, finance, and work",
    "question": "ما حكم أخذ قرض بفائدة لشراء منزل؟"
  },
  {
    "question_id": 154,
    "category": "Business, finance, and work",
    "question": "ما حكم شراء سيارة بالتقسيط بسعر أعلى من النقد؟"
  },
  {
    "question_id": 155,
    "category": "Business, finance, and work",
    "question": "ما شروط البيع بالتقسيط الجائز؟"
  },
  {
    "question_id": 156,
    "category": "Business, finance, and work",
    "question": "ما حكم بيع الذهب بالتقسيط؟"
  },
  {
    "question_id": 157,
    "category": "Business, finance, and work",
    "question": "ما حكم تداول العملات عبر منصات الفوركس؟"
  },
  {
    "question_id": 158,
    "category": "Business, finance, and work",
    "question": "ما حكم الاستثمار في الأسهم؟"
  },
  {
    "question_id": 159,
    "category": "Business, finance, and work",
    "question": "كيف يعرف المسلم أن أسهم شركة ما مباحة؟"
  },
  {
    "question_id": 160,
    "category": "Business, finance, and work",
    "question": "ما حكم الاستثمار في شركة نشاطها مباح لكنها تقترض بالربا؟"
  },
  {
    "question_id": 161,
    "category": "Business, finance, and work",
    "question": "ما حكم العملات الرقمية والتداول بها؟"
  },
  {
    "question_id": 162,
    "category": "Business, finance, and work",
    "question": "ما حكم المضاربة بالرافعة المالية؟"
  },
  {
    "question_id": 163,
    "category": "Business, finance, and work",
    "question": "ما حكم التأمين التجاري؟"
  },
  {
    "question_id": 164,
    "category": "Business, finance, and work",
    "question": "ما حكم التأمين الصحي الذي توفره جهة العمل؟"
  },
  {
    "question_id": 165,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل في شركة تبيع بعض المنتجات المحرمة؟"
  },
  {
    "question_id": 166,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل في متجر يبيع السجائر؟"
  },
  {
    "question_id": 167,
    "category": "Business, finance, and work",
    "question": "ما حكم أخذ عمولة مقابل الوساطة في البيع؟"
  },
  {
    "question_id": 168,
    "category": "Business, finance, and work",
    "question": "ما حكم التسويق بالعمولة عبر الإنترنت؟"
  },
  {
    "question_id": 169,
    "category": "Business, finance, and work",
    "question": "ما حكم التسويق الشبكي؟"
  },
  {
    "question_id": 170,
    "category": "Business, finance, and work",
    "question": "ما حكم البيع قبل تملك السلعة؟"
  },
  {
    "question_id": 171,
    "category": "Business, finance, and work",
    "question": "ما حكم بيع شيء لا يستطيع البائع تسليمه؟"
  },
  {
    "question_id": 172,
    "category": "Business, finance, and work",
    "question": "ما حكم بيع السلع مع إخفاء العيوب؟"
  },
  {
    "question_id": 173,
    "category": "Business, finance, and work",
    "question": "ما حكم الغش في التجارة؟"
  },
  {
    "question_id": 174,
    "category": "Business, finance, and work",
    "question": "ما حكم الاحتكار ورفع الأسعار وقت الأزمات؟"
  },
  {
    "question_id": 175,
    "category": "Business, finance, and work",
    "question": "ما حكم تأخير دفع أجرة العامل؟"
  },
  {
    "question_id": 176,
    "category": "Business, finance, and work",
    "question": "هل يجوز لصاحب العمل خصم جزء من راتب الموظف بسبب خطأ؟"
  },
  {
    "question_id": 177,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل في تصميم إعلانات لمنتجات محرمة؟"
  },
  {
    "question_id": 178,
    "category": "Business, finance, and work",
    "question": "ما حكم استخدام برامج مقرصنة في العمل؟"
  },
  {
    "question_id": 179,
    "category": "Business, finance, and work",
    "question": "ما حكم نسخ الدورات المدفوعة ونشرها مجاناً؟"
  },
  {
    "question_id": 180,
    "category": "Business, finance, and work",
    "question": "ما حكم أخذ أجر على تعليم القرآن؟"
  },
  {
    "question_id": 181,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل الحر مع عميل لا يلتزم بالشروط؟"
  },
  {
    "question_id": 182,
    "category": "Business, finance, and work",
    "question": "ما حكم كتابة الأبحاث الجامعية للآخرين مقابل المال؟"
  },
  {
    "question_id": 183,
    "category": "Business, finance, and work",
    "question": "ما حكم الكذب في السيرة الذاتية للحصول على وظيفة؟"
  },
  {
    "question_id": 184,
    "category": "Business, finance, and work",
    "question": "ما حكم العمل في مجال الذكاء الاصطناعي إذا كان بعض استخداماته محرماً؟"
  },
  {
    "question_id": 185,
    "category": "Business, finance, and work",
    "question": "ما حكم استخدام الذكاء الاصطناعي لإنجاز واجب دراسي؟"
  },
  {
    "question_id": 186,
    "category": "Business, finance, and work",
    "question": "ما حكم المسابقات التي تشترط دفع رسوم للدخول والفوز بجائزة؟"
  },
  {
    "question_id": 187,
    "category": "Business, finance, and work",
    "question": "ما حكم الجوائز التي تقدمها البنوك لعملائها؟"
  },
  {
    "question_id": 188,
    "category": "Business, finance, and work",
    "question": "ما حكم الاقتراض من شخص ثم رد مبلغ زائد دون شرط سابق؟"
  },
  {
    "question_id": 189,
    "category": "Business, finance, and work",
    "question": "ما حكم بيع الدين لشخص آخر؟"
  },
  {
    "question_id": 190,
    "category": "Business, finance, and work",
    "question": "ما حكم العربون في البيع؟"
  },
  {
    "question_id": 191,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم أكل اللحوم المستوردة من بلاد غير إسلامية؟"
  },
  {
    "question_id": 192,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم أكل ذبائح أهل الكتاب؟"
  },
  {
    "question_id": 193,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الأطعمة التي تحتوي على جيلاتين مجهول المصدر؟"
  },
  {
    "question_id": 194,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم استعمال المنتجات التي تحتوي على نسبة من الكحول؟"
  },
  {
    "question_id": 195,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم تناول الأدوية التي تحتوي على الكحول؟"
  },
  {
    "question_id": 196,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم أكل الحشرات؟"
  },
  {
    "question_id": 197,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم أكل لحم الضبع؟"
  },
  {
    "question_id": 198,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم أكل الحيوانات المفترسة؟"
  },
  {
    "question_id": 199,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الذبح الآلي في المسالخ؟"
  },
  {
    "question_id": 200,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الأكل في أواني الذهب والفضة؟"
  },
  {
    "question_id": 201,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الإسراف في الطعام والشراب؟"
  },
  {
    "question_id": 202,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم إلقاء الطعام الصالح في القمامة؟"
  },
  {
    "question_id": 203,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم لبس الذهب للرجال؟"
  },
  {
    "question_id": 204,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم لبس الحرير الطبيعي للرجال؟"
  },
  {
    "question_id": 205,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم إسبال الثوب للرجال؟"
  },
  {
    "question_id": 206,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم لبس الملابس التي عليها صور ذوات الأرواح؟"
  },
  {
    "question_id": 207,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم لبس الملابس التي تحمل شعارات دينية لغير المسلمين؟"
  },
  {
    "question_id": 208,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم حلق اللحية؟"
  },
  {
    "question_id": 209,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم تقصير اللحية وأخذ شيء منها؟"
  },
  {
    "question_id": 210,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم صبغ الشعر بالسواد؟"
  },
  {
    "question_id": 211,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الوشم الدائم؟"
  },
  {
    "question_id": 212,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم إزالة شعر الوجه للمرأة؟"
  },
  {
    "question_id": 213,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم النمص؟"
  },
  {
    "question_id": 214,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم وصل الشعر أو استعمال الشعر المستعار؟"
  },
  {
    "question_id": 215,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم لبس العدسات الملونة للزينة؟"
  },
  {
    "question_id": 216,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم لبس المرأة لباساً ضيقاً أمام النساء؟"
  },
  {
    "question_id": 217,
    "category": "Food, clothing, and daily conduct",
    "question": "ما شروط الحجاب الشرعي للمرأة؟"
  },
  {
    "question_id": 218,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم وضع العطر للمرأة عند الخروج؟"
  },
  {
    "question_id": 219,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم مصافحة المرأة الأجنبية؟"
  },
  {
    "question_id": 220,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الاختلاط في الدراسة والعمل؟"
  },
  {
    "question_id": 221,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الغيبة والنميمة؟"
  },
  {
    "question_id": 222,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم نقل الكلام للإصلاح بين الناس؟"
  },
  {
    "question_id": 223,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الكذب للمزاح؟"
  },
  {
    "question_id": 224,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم المزاح الذي يتضمن تخويف المسلم؟"
  },
  {
    "question_id": 225,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم تصوير الأشخاص بالهاتف؟"
  },
  {
    "question_id": 226,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم رسم ذوات الأرواح؟"
  },
  {
    "question_id": 227,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم اقتناء الصور للذكرى؟"
  },
  {
    "question_id": 228,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الاستماع إلى الموسيقى والأغاني؟"
  },
  {
    "question_id": 229,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم مشاهدة الأفلام والمسلسلات التي تحتوي على محرمات؟"
  },
  {
    "question_id": 230,
    "category": "Food, clothing, and daily conduct",
    "question": "ما حكم الألعاب الإلكترونية التي تحتوي على موسيقى أو صور محرمة؟"
  },
  {
    "question_id": 231,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم قراءة القرآن دون وضوء من الهاتف؟"
  },
  {
    "question_id": 232,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "هل يجوز للجنب قراءة القرآن من الهاتف؟"
  },
  {
    "question_id": 233,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم مس المصحف دون وضوء؟"
  },
  {
    "question_id": 234,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم قراءة القرآن للحائض؟"
  },
  {
    "question_id": 235,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم سجود التلاوة للقارئ والمستمع؟"
  },
  {
    "question_id": 236,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم ختم القرآن في أقل من ثلاثة أيام؟"
  },
  {
    "question_id": 237,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم تخصيص يوم معين لقراءة سورة الكهف؟"
  },
  {
    "question_id": 238,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم قراءة سورة يس لقضاء الحوائج؟"
  },
  {
    "question_id": 239,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "هل ثبت فضل قراءة سورة الملك كل ليلة؟"
  },
  {
    "question_id": 240,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم الاجتماع على قراءة القرآن بصوت واحد؟"
  },
  {
    "question_id": 241,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم أخذ الأجرة على تعليم القرآن؟"
  },
  {
    "question_id": 242,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم تفسير القرآن بالرأي دون علم؟"
  },
  {
    "question_id": 243,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "كيف يتعامل المسلم مع الأحاديث الضعيفة؟"
  },
  {
    "question_id": 244,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "هل يجوز العمل بالحديث الضعيف في فضائل الأعمال؟"
  },
  {
    "question_id": 245,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "كيف يعرف المسلم صحة الحديث؟"
  },
  {
    "question_id": 246,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم نشر حديث قبل التحقق من صحته؟"
  },
  {
    "question_id": 247,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم رواية الحديث بالمعنى؟"
  },
  {
    "question_id": 248,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم الكذب على النبي صلى الله عليه وسلم ولو على سبيل المزاح؟"
  },
  {
    "question_id": 249,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم الاحتفال بالمولد النبوي؟"
  },
  {
    "question_id": 250,
    "category": "Quran, hadith, knowledge, and worship",
    "question": "ما حكم تخصيص ليلة النصف من شعبان بعبادة معينة؟"
  }
]
    questions_df = pd.DataFrame(raw_questions)
    questions_data = raw_questions

print(f"\n✓ Loaded exactly {len(questions_data)} questions across {questions_df['category'].nunique()} categories:")
for cat, cnt in questions_df['category'].value_counts().items():
    print(f"  • {cat:40s}: {cnt:2d} questions")

print("\nSample questions:")
for q in questions_data[:5]:
    print(f"  [{q['question_id']:3d}] ({q['category']}): {q['question']}")


In [ ]:
# 4. Load Qwen 0.6B Embedder and Mizan Reranker on GPU
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDER_NAME = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_NAME = "ALJIACHI/Mizan-Rerank-V2"

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"1. Loading Qwen embedder ({EMBEDDER_NAME}) on [{device}]...")
embedder = SentenceTransformer(EMBEDDER_NAME, device=device)
embedder.max_seq_length = 1024

print(f"2. Loading Mizan reranker ({RERANKER_NAME}) on [{device}]...")
reranker = CrossEncoder(RERANKER_NAME, max_length=1024, device=device, trust_remote_code=True)

# Pre-normalize corpus embeddings on GPU for lightning-fast dot product
print("3. Preparing GPU corpus index...")
corpus_embs_gpu = corpus_embeddings.to(device).float()
corpus_embs_gpu = torch.nn.functional.normalize(corpus_embs_gpu, p=2, dim=1)
print("✓ Models and index ready on GPU!")


In [ ]:
# 5. Execute 6,000 Candidate Generation (250 Questions x 24 Candidates/Q)
# Quotas per question:
#  • Ibn Baz:                 Top 8  (Ranks 1..4 = Official [1,000], Ranks 5..8 = Backup [1,000])
#  • Ibn Uthaymeen:           Top 4  (Ranks 1..2 = Official [500],   Ranks 3..4 = Backup [500])
#  • Al-Lajnah Ad-Da'imah:     Top 4  (Ranks 1..2 = Official [500],   Ranks 3..4 = Backup [500])
#  • Saleh Al-Fawzan:         Top 2  (Rank 1 = Official [250],       Rank 2 = Backup [250])
#  • Abdullah al-Manee':      Top 2  (Rank 1 = Official [250],       Rank 2 = Backup [250])
#  • Abdulaziz Aal ash-Sheikh: Top 2  (Rank 1 = Official [250],       Rank 2 = Backup [250])
#  • Saleh bin Humaid:        Top 2  (Rank 1 = Official [250],       Rank 2 = Backup [250])
# Total per question: 24 candidates (12 official + 12 backup) -> 6,000 candidates total.

SCHOLAR_QUOTAS = {
    "Ibn Baz": {"official": 4, "backup": 4, "total": 8},
    "Ibn Uthaymeen": {"official": 2, "backup": 2, "total": 4},
    "Al-Lajnah Ad-Daimah": {"official": 2, "backup": 2, "total": 4},
    "Saleh Al-Fawzan": {"official": 1, "backup": 1, "total": 2},
    "Abdullah al-Manee": {"official": 1, "backup": 1, "total": 2},
    "Abdulaziz Aal ash-Sheikh": {"official": 1, "backup": 1, "total": 2},
    "Saleh bin Humaid": {"official": 1, "backup": 1, "total": 2},
}

# Step A: Encode all 250 queries
queries_text = [q["question"] for q in questions_data]
print(f"Encoding {len(queries_text)} queries on GPU with Qwen 0.6B...")
query_embs = embedder.encode(queries_text, batch_size=32, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
query_embs = query_embs.to(corpus_embs_gpu.dtype)

# Step B: Compute similarity matrix (250 x 40,062)
print("Computing global vector similarities...")
similarity_matrix = torch.matmul(query_embs, corpus_embs_gpu.T).cpu().numpy()

# Pre-index corpus rows by scholar
scholar_indices = {}
for scholar in SCHOLAR_QUOTAS:
    scholar_indices[scholar] = np.where(corpus_df["source"].values == scholar)[0]

# Step C: Retrieve & Rerank
all_records = []
official_records = []
backup_records = []

print(f"\nRetrieving & Mizan-reranking candidate fatwas across 250 questions...")
t0 = time.time()

for q_idx, q_item in enumerate(tqdm(questions_data, desc="Generating Candidates")):
    q_id = q_item["question_id"]
    q_text = q_item["question"]
    q_cat = q_item["category"]
    
    q_sims = similarity_matrix[q_idx]
    
    # For each scholar, collect initial candidate pool and rerank
    for scholar, quota in SCHOLAR_QUOTAS.items():
        idx_pool = scholar_indices.get(scholar)
        if idx_pool is None or len(idx_pool) == 0:
            continue
            
        # Pool size: take top (quota['total'] * 3) by vector similarity
        pool_size = min(len(idx_pool), max(quota["total"] * 3, 10))
        top_sim_subindices = np.argpartition(-q_sims[idx_pool], pool_size - 1)[:pool_size]
        top_corpus_indices = idx_pool[top_sim_subindices]
        
        # Prepare pairs for Mizan reranker
        cand_rows = corpus_df.iloc[top_corpus_indices]
        pairs = []
        for _, r in cand_rows.iterrows():
            # Construct reranker text (title + excerpt of answer)
            ans_part = str(r['answer'])[:700]
            text_pair = f"{r['title']} | {ans_part}"
            pairs.append([q_text, text_pair])
            
        # Predict reranker scores
        rerank_scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
        
        # Sort candidates descending by Mizan score
        sorted_order = np.argsort(-rerank_scores)
        target_total = min(quota["total"], len(sorted_order))
        
        for rank_num, order_idx in enumerate(sorted_order[:target_total], 1):
            c_row = cand_rows.iloc[order_idx]
            r_score = float(rerank_scores[order_idx])
            emb_score = float(q_sims[top_corpus_indices[order_idx]])
            
            # Determine split (official top half vs backup second half)
            split = "official" if rank_num <= quota["official"] else "backup"
            
            record = {
                "id": len(all_records) + 1,
                "question_id": q_id,
                "category": q_cat,
                "question": q_text,
                "scholar": scholar,
                "fatwa_id": int(c_row["fatwa_id"]),
                "title": str(c_row["title"]),
                "question_text": str(c_row.get("question", "")),
                "answer_text": str(c_row["answer"]),
                "reranker_score": round(r_score, 4),
                "embedding_score": round(emb_score, 4),
                "rank_in_scholar": rank_num,
                "dataset_split": split,
            }
            
            all_records.append(record)
            if split == "official":
                official_records.append(record)
            else:
                backup_records.append(record)

dt_total = time.time() - t0
print(f"\n✓ Finished in {dt_total:.1f}s ({len(all_records)/(dt_total+1e-5):.1f} items/s).")
print(f"✓ Total candidates generated: {len(all_records):,}")
print(f"  • Official Candidates (Main 3K): {len(official_records):,}")
print(f"  • Backup Candidates (Reserve 3K): {len(backup_records):,}")


In [ ]:
# 6. Export Datasets & Create Download Bundle
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.DataFrame(all_records)
df_official = pd.DataFrame(official_records)
df_backup = pd.DataFrame(backup_records)

# Save CSVs
df_official.to_csv(OUTPUT_DIR / "candidates_official_3k.csv", index=False, encoding="utf-8")
df_backup.to_csv(OUTPUT_DIR / "candidates_backup_3k.csv", index=False, encoding="utf-8")
df_all.to_csv(OUTPUT_DIR / "candidates_all_6k.csv", index=False, encoding="utf-8")

# Save JSONs
df_official.to_json(OUTPUT_DIR / "candidates_official_3k.json", orient="records", force_ascii=False, indent=2)
df_backup.to_json(OUTPUT_DIR / "candidates_backup_3k.json", orient="records", force_ascii=False, indent=2)
df_all.to_json(OUTPUT_DIR / "candidates_all_6k.json", orient="records", force_ascii=False, indent=2)

# Create ZIP archive for one-click Kaggle download
zip_path = OUTPUT_DIR / "candidates_6k_bundle.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_DIR / "candidates_official_3k.csv", arcname="candidates_official_3k.csv")
    zf.write(OUTPUT_DIR / "candidates_official_3k.json", arcname="candidates_official_3k.json")
    zf.write(OUTPUT_DIR / "candidates_backup_3k.csv", arcname="candidates_backup_3k.csv")
    zf.write(OUTPUT_DIR / "candidates_backup_3k.json", arcname="candidates_backup_3k.json")
    zf.write(OUTPUT_DIR / "candidates_all_6k.csv", arcname="candidates_all_6k.csv")
    zf.write(OUTPUT_DIR / "candidates_all_6k.json", arcname="candidates_all_6k.json")

print("=" * 80)
print("FILES SAVED TO KAGGLE WORKING DIRECTORY:")
print("=" * 80)
for f in OUTPUT_DIR.glob("candidates_*"):
    sz_mb = f.stat().st_size / (1024**2)
    print(f"  • {f.name:32s} ({sz_mb:.2f} MB)")

print("\n" + "=" * 80)
print("OFFICIAL 3K POOL DISTRIBUTION BY SCHOLAR:")
print("=" * 80)
print(df_official["scholar"].value_counts().to_string())

print("\n" + "=" * 80)
print("BACKUP 3K POOL DISTRIBUTION BY SCHOLAR:")
print("=" * 80)
print(df_backup["scholar"].value_counts().to_string())

print("\n✓ All done! Download candidates_6k_bundle.zip and proceed to local labeling with GPT-4o Mini.")
